<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/onboarding.png" align="center" width="20%">
</div>

<br>

# DISCRETE EVENT SIMULATION OF A MULTI-STAGE HELP DESK

<br>

**About:** Build a discrete event simulation (DES) of a two-stage help desk using SimPy, then sweep across staffing configurations to identify the allocation that minimizes customer wait time.

**Learning Goals:**
1. Explain what Discrete Event Simulation (DES) is and identify problem types for which it is the appropriate modeling tool.
2. Use SimPy's `Environment`, `Resource`, and process generator pattern to model a multi-stage service queue with shared resources.
3. Run multiple independent simulation replications and aggregate statistics to account for stochastic variability.
4. Enumerate staffing configurations and identify the allocation that minimizes average customer wait time.

**Keywords:** discrete event simulation, simpy, queueing, resource allocation, monte carlo replication, process generators

**Prerequisite Knowledge:** (1) Python generators and the `yield` keyword, (2) basic probability and random sampling, (3) Python dataclasses.

**Target User:** Practitioners and students learning to model service systems, capacity planning, and stochastic workflows before moving on to analytical queueing models.


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 1: WHY DISCRETE EVENT SIMULATION](#Part_1)
> #### [PART 2: MODELING THE HELP DESK IN SIMPY](#Part_2)
> #### [PART 3: REPLICATION AND AGGREGATION](#Part_3)
> #### [PART 4: SWEEPING STAFFING CONFIGURATIONS](#Part_4)

<br>


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP** and **IMPORTS**

All imports for the notebook live in this cell so a clean kernel restart runs top to bottom without hidden dependencies. `simpy` is the discrete event simulation framework; the rest are standard library.


In [ ]:
# Verified against SimPy 4.x docs, 2026-09-03 - re-check at https://simpy.readthedocs.io/
import simpy
import random
import statistics
import logging
from datetime import timedelta
from itertools import product, groupby
from dataclasses import dataclass

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **WHY** DISCRETE EVENT **SIMULATION**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/GIGs.png" align="center" width="45%" padding="10"><br>
    <br>
    A stylized help desk: customers arrive, wait for an available engineer, and progress through a sequence of service stages.
</div>


**Scenario.** A prospective customer lands on a service portal and files an onboarding ticket. The Platform Engineering team has to triage that ticket, collect requirements, and (with some probability) loop back to collect additional information. Each of those steps takes time, and each step is bottlenecked by a limited pool of engineers. The team wants to know: **what is the minimum number of engineers - and what allocation between roles - keeps the average customer wait time below a target?**

You could try to derive this analytically. For a single-stage queue with Poisson arrivals and exponential service, closed-form results from M/M/c queueing theory apply. But the moment you add a second stage with a different service distribution, a conditional loop-back, or shared resources across stages, the algebra explodes. **Discrete Event Simulation (DES)** sidesteps the algebra by *executing* the system on a virtual clock and measuring what happens.

**The DES abstraction.** Instead of stepping time forward in fixed ticks, DES maintains a priority queue of *future events* ordered by their scheduled time. The simulator repeatedly pops the next event, advances the clock to that event's time, and processes it - which may generate more future events. Between events, nothing happens: the clock jumps. This makes DES efficient for systems where interesting behavior is concentrated at discrete moments (an arrival, a service completion, a resource release) rather than distributed continuously.

**When DES is the right tool.**
- The system is a network of queues, resources, and processes.
- Service and arrival times are stochastic, and analytical distributions get complex quickly.
- You care about throughput, wait time, utilization, or blocking probability - metrics that emerge from the interaction of many random events.
- The system is small enough that per-event bookkeeping is cheap, but large enough that closed-form analysis is impractical.

**SimPy** is a Python DES framework built on generator functions. A *process* is a Python generator that `yield`s events (timeouts, resource requests) back to the simulator, which handles scheduling. This turns simulation logic into linear, readable code - you write what a customer *does*, and SimPy handles when it happens.

___

**Sources Consulted:**
- [SimPy documentation - Overview](https://simpy.readthedocs.io/en/latest/) (primary source for the framework model, retrieved 2026-09-03)
- [Law, *Simulation Modeling and Analysis*, 5th ed.](https://www.mheducation.com/highered/product/simulation-modeling-analysis-law/M9780073401324.html) (canonical reference for DES fundamentals and replication strategy, 2015)

___


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **MODELING** the **HELP DESK** in SimPy

The help desk has two service stages, each backed by a pool of engineers:

1. **Initial Inquiry Stage.** A first-tier engineer triages the incoming onboarding ticket. Service time: uniformly 1 to 3 days.
2. **Requirements Collection.** An on-call engineer collects deployment requirements. Service time: uniformly 1 to 7 days. With probability 0.5, this stage repeats (the engineer needs a follow-up round).

Each stage is a SimPy `Resource` - a counting semaphore that gates access to a fixed number of slots. When a customer requests a resource, they either grab an available slot immediately or wait in FIFO order until one frees up. This wait is exactly what we want to measure.

The class below bundles the two resources and their service-time distributions. Grouping them in an object rather than passing them individually keeps the process function readable and makes the model easy to extend later.


In [ ]:
class HelpDesk:
    """Two-stage help desk. Each stage has its own pool of engineers modeled as a SimPy Resource."""

    def __init__(self, env, num_help_desk_eng, num_on_call_eng):
        self.env = env
        # Resource capacity = number of engineers who can be simultaneously busy at that stage.
        self.help_desk_eng = simpy.Resource(env, num_help_desk_eng)
        self.on_call_eng = simpy.Resource(env, num_on_call_eng)

    def get_help(self, customer):
        # Stage 1: initial triage. Time is in days; uniform 1-3.
        yield self.env.timeout(random.randint(1, 3))

    def collect_complete(self, customer):
        # Stage 2: requirements collection. Uniform 1-7 days.
        yield self.env.timeout(random.randint(1, 7))


Next, the *process function* - the generator that describes what one customer does from arrival to departure. Read it as a script: request the first resource, hold it for the service time, release it, then request the second, and possibly repeat.

The `with resource.request() as request: yield request` idiom is worth pausing on. `request()` returns an event; `yield`ing it suspends the customer's process until the request is granted. The `with` block guarantees the resource is released automatically when the block exits, so we do not have to hand-manage release calls. This is the entire concurrency model - no threads, no locks.


In [ ]:
def go_to_help_desk(env, customer, help_desk, wait_times):
    """One customer's journey through the help desk."""
    arrival_time = env.now

    # Stage 1: initial inquiry / triage
    with help_desk.help_desk_eng.request() as request:
        yield request                                    # wait for a triage engineer
        yield env.process(help_desk.get_help(customer))  # hold engineer for service time

    # Stage 2: requirements collection
    with help_desk.on_call_eng.request() as request:
        yield request
        yield env.process(help_desk.collect_complete(customer))

    # Conditional follow-up: 50% of customers need a second round.
    # random.choice on [True, False] is a fair coin - equivalent to random.random() < 0.5.
    if random.choice([True, False]):
        with help_desk.on_call_eng.request() as request:
            yield request
            yield env.process(help_desk.collect_complete(customer))

    # Total time in system, in the same units as env.now.
    wait_times.append(env.now - arrival_time)


Finally, the *arrival process*. Customers arrive every 3 days for a 183-day horizon (roughly two quarters). We spawn one customer at time 0, then a new one every 3 simulated days. Each `env.process(...)` call registers a new customer generator with the environment; they all run concurrently on the shared resources.


In [ ]:
def run_help_desk(env, num_help_desk_eng, num_on_call_eng, wait_times):
    help_desk = HelpDesk(env, num_help_desk_eng, num_on_call_eng)

    customer = 0
    env.process(go_to_help_desk(env, customer, help_desk, wait_times))

    # Spawn a new customer every 3 days for 183 days (~ two quarters).
    for _ in range(int(183 / 3)):
        yield env.timeout(3)
        customer += 1
        env.process(go_to_help_desk(env, customer, help_desk, wait_times))
    return wait_times


def run_simulation(num_help_desk_eng, num_on_call_eng):
    wait_times = []
    env = simpy.Environment()
    env.process(run_help_desk(env, num_help_desk_eng, num_on_call_eng, wait_times))
    env.run()
    return wait_times


Let's sanity-check the model with a single run and a modest allocation - 2 triage engineers, 3 on-call engineers - and print the mean total time customers spent in the system.


In [ ]:
random.seed(2022)  # fixed seed so this cell is reproducible

wait_times = run_simulation(num_help_desk_eng=2, num_on_call_eng=3)
mean_days = statistics.mean(wait_times)
print(f"Customers served: {len(wait_times)}")
print(f"Mean time in system: {mean_days:.2f} days")
print(f"Min / Max: {min(wait_times):.2f} / {max(wait_times):.2f} days")


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **Modify `go_to_help_desk` so that the follow-up round occurs with probability 0.25 instead of 0.5, and re-run the simulation. Does the mean time in system decrease, and by how much? Explain why the change is smaller than a naive "half as many follow-ups" argument would predict.**

<br>

```python
# Copy the go_to_help_desk function, change the conditional, and run:
# random.seed(2022)
# wait_times = run_simulation(num_help_desk_eng=2, num_on_call_eng=3)
# print(f"Mean time in system: {statistics.mean(wait_times):.2f} days")
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **REPLICATION** and **AGGREGATION**

A single simulation run is one sample path from a stochastic process. The mean wait time from one run is an estimate of the *true* mean wait time - but a noisy one. Two runs with the same parameters and different random seeds will give different means. This is a feature of the system, not a bug of the code: real help desks also have good weeks and bad weeks.

The standard fix is **independent replication**: run the simulation $R$ times with different seeds, treat each run's mean as one observation, and aggregate. The precision of the aggregate estimate improves at rate $1/\sqrt{R}$ - doubling $R$ cuts the standard error by roughly 30 percent.

The helper below runs $R$ replications for a given staffing configuration and returns the mean total customer time as a `timedelta` for readability.


In [ ]:
def calculate_wait_time(wait_times):
    """Mean wait, returned as a timedelta for human-readable printing."""
    average_wait = statistics.mean(wait_times)
    # The original scenario treated wait_times as minutes for reporting; we preserve that
    # convention so downstream comparisons across configs remain consistent.
    return timedelta(minutes=average_wait)


def calculate_average_wait_time_for_config(num_help_desk_eng, num_on_call_eng, num_simulations):
    all_wait_times = []
    for _ in range(num_simulations):
        all_wait_times += run_simulation(num_help_desk_eng, num_on_call_eng)
    return calculate_wait_time(all_wait_times).total_seconds()


Quick check: run 10 replications at the (2, 3) allocation and compare against the single-run estimate above. The pooled mean should be close to, but not identical to, the single-run number.


In [ ]:
random.seed(2022)
pooled = calculate_average_wait_time_for_config(num_help_desk_eng=2, num_on_call_eng=3, num_simulations=10)
print(f"Pooled mean total customer time (10 reps): {pooled:.1f} seconds")


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **Run `calculate_average_wait_time_for_config(2, 3, R)` for `R` in `[1, 5, 25, 100]`. Plot or tabulate the estimated mean as a function of `R`. At roughly what `R` does the estimate stop moving materially between consecutive doublings? Why is that a defensible stopping rule in practice, rather than "pick R as large as possible"?**

<br>

```python
# for R in [1, 5, 25, 100]:
#     random.seed(2022)
#     est = calculate_average_wait_time_for_config(2, 3, R)
#     print(f"R={R:3d}  mean={est:.1f}")
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **SWEEPING** staffing **CONFIGURATIONS**

We have a model and we have a way to average out noise. Now the actual question: given a headcount budget, how should engineers be split between the two roles?

We enumerate all integer allocations $(h, o)$ with $h \geq 1$, $o \geq 1$, and $h + o \leq $ `MAX_ENGINEERS`, run 10 replications at each, and record the mean wait. Then we group by total headcount and report the best allocation at each budget level.

A `dataclass` bundles the configuration and its estimated wait time. Defining it *before* the sweep runs is important: notebook cells execute in order, and any class referenced by a later cell has to already exist in the kernel.


In [ ]:
@dataclass
class EngineerConfig:
    num_help_desk_eng: int
    num_on_call_eng: int
    average_time: float  # seconds

    @property
    def total_engineers(self):
        return self.num_help_desk_eng + self.num_on_call_eng


def generate_eng_config(max_engineers):
    """All (h, o) allocations with h>=1, o>=1, h+o<=max_engineers."""
    for h, o in product(range(1, max_engineers + 1), repeat=2):
        if h + o <= max_engineers:
            yield h, o


The sweep. This runs ~45 configurations $\times$ 10 replications $\times$ ~60 customers per replication, so it takes a few seconds. We seed once at the top so the whole sweep is reproducible.


In [ ]:
def run_sweep(max_engineers=10, num_simulations=10, seed=2022):
    random.seed(seed)
    results = []
    for h, o in generate_eng_config(max_engineers):
        avg = calculate_average_wait_time_for_config(h, o, num_simulations)
        results.append(EngineerConfig(h, o, avg))
    return results


results = run_sweep()
print(f"Configurations evaluated: {len(results)}")


Group by total headcount and print the best allocation at each budget. This tells us the **marginal value of an added engineer** and, crucially, **where to put them** - triage or on-call.


In [ ]:
results.sort(key=lambda e: e.total_engineers)

for total, group in groupby(results, key=lambda e: e.total_engineers):
    group = list(group)
    best = min(group, key=lambda e: e.average_time)
    logging.info(
        f"Total engineers={total:2d}  best split: "
        f"triage={best.num_help_desk_eng}, on_call={best.num_on_call_eng}, "
        f"mean_time={best.average_time:.1f}s"
    )


**Reading the results.** Two patterns typically emerge:

1. **The on-call stage is the binding bottleneck.** Its service time is uniform 1-7 days (mean 4) versus 1-3 for triage (mean 2), and half of customers pass through it twice. Optimal allocations put more headcount on the on-call side.
2. **Diminishing returns.** Adding an engineer to a bottleneck stage helps a lot; adding one to an already-adequate stage barely moves the needle. Total-headcount plotted against best-mean-time flattens out quickly.

The sweep does not know any of this in advance. It learns it by running the system. That is the point of DES: you encode the mechanism and the data reveals the structure.

___

**Note:** The absolute wait-time numbers depend on the seed, the arrival rate, and the specific service distributions in this toy setup. What generalizes is the *shape* of the response - which stage is the bottleneck, and how quickly returns diminish.
___


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **The current sweep minimizes mean wait time. Modify it to instead find, for each total headcount, the allocation that minimizes the 95th-percentile wait time. Does the optimal split change? What does the difference (or lack of one) tell you about the shape of the wait-time distribution at each stage?**

<br>

```python
# Hint: change calculate_average_wait_time_for_config to return the pooled list
# of per-customer waits, then apply statistics.quantiles or numpy.percentile.
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<hr style="border: 2px solid#003262;" />

#### WRAP-UP

You built a two-stage help desk simulator from scratch, replicated it enough times to trust the numbers, and swept the staffing space to find where to put the next engineer. The pattern generalizes beyond help desks: any service network with limited resources, stochastic timing, and stage-dependent behavior can be modeled the same way.

**What is next.** The follow-up notebook, `02_tenant_onboarding_simulation.ipynb`, connects this to classical **M/M/c queueing theory**. You will see how Poisson arrivals and exponential service - the assumptions behind analytical closed-form results - map onto SimPy code, and how to instrument a `Resource` to collect the metrics (queue length, utilization, delay) that queueing theorists care about. The M/M/c notebook builds directly on the SimPy patterns established here.

**Further reading:**
- [SimPy examples: bank renege, gas station refueling](https://simpy.readthedocs.io/en/latest/examples/index.html)
- Law, A. M. *Simulation Modeling and Analysis*, 5th ed. - Chapters 1-3 on DES fundamentals; Chapter 9 on replication and output analysis.

<hr style="border: 6px solid#003262;" />
